# Tech Challenge 4

O Desafio
Você e seu grupo foram contratados para atuar como Data Analysts em
uma empresa de e-commerce.

A empresa deseja compreender melhor o desempenho de seu portfólio e a
percepção dos consumidores, utilizando informações sobre:

<ul>
<li>Produtos E Categorias;</li>
<li>Preços E Descontos;</li>
<li>Avaliações E Quantidade De Reviews;</li>
<li>Descrições Dos Produtos;</li>
<li>Comentários Realizados Pelos Consumidores</li>
</ul>

A análise deverá buscar responder à seguinte questão:

Quais produtos, categorias e estratégias comerciais merecem maior atenção
e quais oportunidades podem ser identificadas a partir dos dados
disponíveis?

A liderança da empresa deseja compreender melhor a relação entre:

Preço → Desconto → Engajamento → Satisfação do Cliente

Algumas perguntas que podem orientar o desenvolvimento são:
<ul>
<li>Quais categorias e produtos apresentam maior relevância?</li>
<li>Quais produtos possuem maior engajamento dos consumidores?</li>
<li>Existe relação entre descontos e avaliações?</li>
<li>Quais categorias apresentam maior ou menor satisfação?</li>
<li>Existem produtos com alto engajamento, mas baixa avaliação?</li>
<li>Existem produtos com boa avaliação, mas pouca visibilidade?</li>
<li>Quais padrões podem ser identificados nas avaliações dos
consumidores?</li>
<li>Quais oportunidades ou riscos podem ser identificados para o negócio?</li>
</ul>

A solução deverá permitir a análise de:

Portfólio: exploração de produtos e categorias para identificar pontos fortes,
oportunidades e produtos que merecem atenção.
Preços e Descontos: análise da relação entre preço original, preço com
desconto e percentual de desconto

A equipe deverá investigar possíveis padrões relacionados ao
comportamento comercial dos produtos.
Engajamento e Satisfação - análise da relação entre:
Quantidade de Avaliações × Satisfação dos Clientes

Customer Insights: além dos indicadores quantitativos, a equipe deverá
explorar os conteúdos das avaliações dos consumidores.

A análise poderá buscar responder:
<ul>
<li>O que os clientes mais valorizam?</li>
<li>Quais problemas aparecem com maior frequência?</li>
<li>Quais padrões estão presentes em avaliações positivas?</li>
<li>Quais temas aparecem em avaliações negativas?</li>
</ul>

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [ ]:
FILENAME = "data\\amazon.csv"
df = pd.read_csv(FILENAME, delimiter=",")
df.info()

In [ ]:
df.head()

## Data Cleaning

<ul>
<li>Padronização da Moeda</li>
<li>Padronização da porcentagem</li>
<li>Padronização do Rating</li>
<li>Padronização da contagem de ratings</li>
<li>Remover duplicados</li>
<li>Segmentar Categorias</li>
</ul>

In [ ]:
df.drop_duplicates()

In [ ]:
"""
Segmentação de categorias
"""

for index, row in df.iterrows():
    temp = row["category"].split("|")
    for i in range(len(temp)):
        df.loc[index, [f"category {i+1}"]] = temp[i]


In [ ]:
label_dict = {}

for i in range(1, 7):
    temp_le = LabelEncoder()
    df[f"category {i} encoded"] = temp_le.fit_transform(df[f"category {i}"])
    label_dict[f"category {i}"] = [cat for cat in temp_le.classes_]


In [ ]:
# Padronização de rating
df["rating"] = df["rating"].replace("|", "0")
df["rating"] = df["rating"].astype(float)

In [ ]:
# Padronização de moeda
df["actual_price"] = (df["actual_price"]
                      .str.replace("₹", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .astype(float))

df["discounted_price"] = (df["discounted_price"]
                      .str.replace("₹", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .astype(float))

In [ ]:
# Padronização da porcentagem
df["discount_percentage"] = (df["discount_percentage"]
                      .str.replace("%", "", regex=False)
                      .str.replace(".", "", regex=False)
                      .astype(float))

df["discount_percentage"] = df["discount_percentage"]/100


In [ ]:
# Padronização do rating count
df["rating_count"] = df["rating_count"].fillna("0,0")
df["rating_count"] = (df["rating_count"]
                      .str.replace(",", "", regex=False)
                      .astype(float))

In [ ]:
df.info()

## Overview Pós Limpeza

In [ ]:
df.describe().T

In [ ]:
print(df["rating"].skew())
sns.histplot(df["rating"], bins=15)
plt.show()

In [ ]:
print(df["actual_price"].skew())
sns.histplot(df["actual_price"])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (8,8))
corr_table = df[df.select_dtypes(include=['float', 'int']).columns]

sns.heatmap(data = corr_table.corr().round(2),
            annot=True,
            linewidths=0.5,
            ax = ax)

## Creating an Model

## Exporting Data for Power BI